# 00.3 Pandas for Modeling

The goal of this notebook is not to cover all of `Pandas`, but to master the most common data-preparation operations used before modeling.

Key concepts:

- `DataFrame`
- `Series`
- groupby and aggregation
- categorical encoding

## Learning Goals

After this notebook, you should be able to:

1. Create and inspect a `DataFrame`.
2. Select columns, filter rows, and use `loc`.
3. Handle missing values.
4. Use `groupby` for basic summaries.
5. Use `get_dummies` for basic categorical encoding.
6. Prepare clean data for downstream models.

In [ ]:
import numpy as np
import pandas as pd

## Creating and Inspecting a `DataFrame`

A `DataFrame` can be thought of as a two-dimensional table with named columns.


In [ ]:
df = pd.DataFrame(
    {
        "student_id": [101, 102, 103, 104, 105, 106],
        "hours": [1.5, 3.0, 2.2, 1.0, 4.1, 2.8],
        "attendance": [0.70, 0.90, 0.80, 0.60, 0.95, 0.85],
        "sleep_hours": [6.0, 7.0, np.nan, 5.5, 7.5, 6.8],
        "city": ["A", "B", "A", "B", "C", None],
        "passed": [0, 1, 1, 0, 1, 1],
    }
)

print("head / head:")
print(df.head())
print()
print("shape / shape:", df.shape)
print("columns / columns:", list(df.columns))

In [ ]:
print("info / info:")
print(df.info())
print()
print("describe / describe:")
print(df.describe(include="all"))

Important idea:

- shows dtypes and missing values
- shows summary statistics
- tells you number of rows and columns

## Selecting Columns and Filtering Rows

Before machine learning, the most common actions are selecting columns and filtering rows.


In [ ]:
feature_cols = ["hours", "attendance", "sleep_hours"]
features = df[feature_cols]
target = df["passed"]

high_attendance = df[df["attendance"] >= 0.85]
selected_rows = df.loc[df["city"].isin(["A", "B"]), ["student_id", "city", "passed"]]

print("feature columns / feature columns:")
print(features)
print()
print("target / target:")
print(target)
print()
print("high attendance rows / high attendance rows:")
print(high_attendance)
print()
print("selected rows and columns / selected rows and columns:")
print(selected_rows)

In [ ]:
# Exercise 1
# Goal:
# select all rows with city == 'A'
# keep only hours and passed
# then keep only rows with hours > 2

# city_a =
# result =

# print(result)

In [ ]:
# Exercise 1 Reference Solution

city_a = df[df["city"] == "A"]
result = city_a.loc[city_a["hours"] > 2, ["hours", "passed"]]
print(result)

## Missing Values

Missing values are extremely common in real datasets.

Before filling missing values, first answer two questions:

1. Which columns are missing?/ Which columns are missing values?
2. Should the missing values be dropped, imputed, or encoded separately?/ Should they be dropped, imputed, or encoded separately?

In [ ]:
print("missing values per column / missing values per column:")
print(df.isna().sum())
print()

df_filled = df.copy()
df_filled["sleep_hours"] = df_filled["sleep_hours"].fillna(df_filled["sleep_hours"].mean())
df_filled["city"] = df_filled["city"].fillna("Unknown")

print("after filling / after filling:")
print(df_filled)
print()
print("missing values after filling / missing values after filling:")
print(df_filled.isna().sum())

Common strategies:

- mean, median, or fixed values
- mode, `Unknown`, or a dedicated category
- you may need to drop the column

In [ ]:
# Exercise 2
# Implement clean_student_df(dataframe)
#Requirements / Requirements:
# copy the input DataFrame
# fill sleep_hours with its mean
# fill city with "Unknown"
# return the cleaned DataFrame

def clean_student_df(dataframe):
    # TODO
    pass


# print(clean_student_df(df))

In [ ]:
# Exercise 2 Reference Solution

def clean_student_df_solution(dataframe):
    result = dataframe.copy()
    result["sleep_hours"] = result["sleep_hours"].fillna(result["sleep_hours"].mean())
    result["city"] = result["city"].fillna("Unknown")
    return result


print(clean_student_df_solution(df))

## Groupby and Aggregation

`groupby` groups rows by one or more columns and then computes summary statistics.

It is very useful during exploratory data understanding.


In [ ]:
city_summary = (
    df_filled.groupby("city")
    .agg(
        avg_hours=("hours", "mean"),
        avg_attendance=("attendance", "mean"),
        pass_rate=("passed", "mean"),
        num_students=("student_id", "count"),
    )
    .sort_values("pass_rate", ascending=False)
)

print("summary by city / summary by city:")
print(city_summary)

Here `pass_rate` is computed by the mean because the label is binary 0/1.

This is a very common small trick in binary classification data analysis.


## Categorical Encoding

Models usually cannot consume raw string categories directly, so they need encoding.

The most basic method is one-hot encoding, often done with `pd.get_dummies`.


In [ ]:
encoded = pd.get_dummies(df_filled, columns=["city"], dtype=int)

print("after encoding / after encoding:")
print(encoded)
print()
print("after encodingcolumns / encoded columns:")
print(list(encoded.columns))

## Minimal Modeling-Oriented Workflow

In practice, what you often really need is:

1. clean the data
2. encode categorical variables
3. split features and target
4. convert to a format a downstream model can use

In [ ]:
model_df = pd.get_dummies(clean_student_df_solution(df), columns=["city"], dtype=float)
X = model_df.drop(columns=["student_id", "passed"])
y = model_df["passed"]

print("X / features:")
print(X)
print()
print("y / target:")
print(y)
print()
print("X shape / X shape:", X.shape)
print("y shape / y shape:", y.shape)

In [ ]:
# Exercise 3
# Implement prepare_features_and_target(dataframe)
#Requirements / Requirements:
# clean missing values first
# one-hot encode city
# drop student_id and passed from features
# return X, y

def prepare_features_and_target(dataframe):
    # TODO
    pass


# X_out, y_out = prepare_features_and_target(df)
# print(X_out)
# print(y_out)

In [ ]:
# Exercise 3 Reference Solution

def prepare_features_and_target_solution(dataframe):
    cleaned = clean_student_df_solution(dataframe)
    encoded = pd.get_dummies(cleaned, columns=["city"], dtype=float)
    X = encoded.drop(columns=["student_id", "passed"])
    y = encoded["passed"]
    return X, y


X_out, y_out = prepare_features_and_target_solution(df)
print(X_out)
print(y_out)

## Summary

The core of this notebook is not memorizing `Pandas` APIs, but learning the minimal workflow before modeling.

You should now be able to answer:

1. What is the relationship between a `DataFrame` and a `Series`?
2. Why check missing values before filling them?
3. Why do we often encode categories before modeling?
4. Why is `groupby` useful for understanding the data?

Suggested next step:

- Move to the visualization notebook and learn to "see" your data and model behavior.